# ДЗ2. Опыт 10: cosine LR у широкой модели

**Согласованное изменение относительно 09: постоянный LR 0.001 → cosine от 0.001 до 0.0001 за 50 000 обновлений, без warmup и рестартов.** Начальный LR сохраняется. Архитектура, данные, tokenizer, budget и генерация прежние: Transformer 2+2/256, FFN1024, 4 головы, ReLU/Post-LN, learned positions2048, dropout0.1; 6 312 960 параметров. Новые веса/AdamW с seed42, прежние20k пар/batch8/FP32/clipping1/greedy KV/cap512. Продолжение checkpoint09 не выполняется.

09 ухудшился относительно08: одинаковый BPE/20k/50k, validation64 loss1.573→1.944 и train64 1.070→1.383.1381/1919 test-вопросов получили одну программу с неопределённым `boys`. На сохранённых весах09 подмена вопросов мало меняет teacher-forcing loss. Причина не доказана;10 проверяет пользу снижения LR при прежнем старте. Если проблема возникает на первых обновлениях, медленный начальный cosine может её не исправить. Warmup и Pre-LN оставлены отдельными последующими факторами.

В новой Colab-сессии включить **T4** и выполнить11 кодовых ячеек по порядку. Точный собственный BPE08 встроен и проверяется поSHA; старые файлы загружать не нужно. Контроли2k/5k/10k/20k/30k/50k. Сравниваем с09 на том же шаге;08 остаётся более успешной предыдущей базой. На10k можно оценить ход опыта, но горизонт scheduler всегда50k — не менять его после промежуточной оценки.

Контроли: loss/EOS/AST/return, ограниченная численная арифметика и ошибки имён, повторы, разнообразие программ; CE с переставленными вопросами при неизменных gold-кодах. Последняя проверка оценивает использование источника при teacher forcing, не правильность решения. Дополнительная диагностика не меняет обучение и исключена из training/generation timers.

Последняя ячейка скачивает **`hw02-bpe4k-width256-cosine-20k-results.zip`**, в том числе после прерывания train. Отдельно скачать этот **IPYNB с outputs**. ZIP содержит results/tokenizer/latest/контрольные checkpoints/test, README и manifest. В checkpoints сохраняются также scheduler и RNG. Встроенные показатели08/09 — сохранённые результаты пользователя, не новые измерения10.

Официальный pass@1 не измеряется: численная AST-проверка поддерживает ограниченный набор программ; ошибки reference возможны. Исходные данные и таргеты сохраняются.

Источники: [неизменное ДЗ2](https://github.com/ashaba1in/hse-nlp/blob/7f837940c4af9a686fca7bff5f02345218b6fe80/homeworks/hw2_code_generation.md), [PyTorch CosineAnnealingLR](https://docs.pytorch.org/docs/stable/generated/torch.optim.lr_scheduler.CosineAnnealingLR.html). Конкретные LR/бюджет — наши экспериментальные параметры, не гарантия улучшения.


In [1]:
%pip install -q tokenizers==0.23.2 pyarrow


## 1. Настройки и сохранение результатов


In [2]:
import ast
import base64
import zlib
import hashlib
import json
import math
import os
import random
import time
import zipfile
from collections import Counter
from pathlib import Path
from urllib.request import urlretrieve

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import tokenizers
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders
import pyarrow.parquet as pq
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence

SEED, TRAIN_SIZE, BATCH_SIZE = 42, 20000, 8
EVALUATION_STEPS = (2000, 5000, 10000, 20000, 30000, 50000)
TOTAL_STEPS, LOG_EVERY, CHECKPOINT_EVERY = 50000, 50, 1000
D_MODEL, N_LAYERS, MAX_POSITIONS = 256, 2, 2048
LEARNING_RATE, MIN_LEARNING_RATE = 0.001, 0.0001
PAD, BOS, EOS, TOKEN_OFFSET, TARGET_VOCAB = 0, 1, 2, 3, 4096
GENERATION_CAP, INFERENCE_BATCH, BYTE_BUDGET = 512, 32, 512
LOGIT_ATOL, LOGIT_RTOL = 1e-4, 1e-4
RUN_DIR = Path('hw02-bpe4k-width256-cosine-20k')
RESULTS_FILE = RUN_DIR / 'results.json'
TOKENIZER_FILE = RUN_DIR / 'tokenizer.json'
LATEST_CHECKPOINT = RUN_DIR / 'latest.pt'
ARCHIVE_FILE = Path('hw02-bpe4k-width256-cosine-20k-results.zip')
assert tokenizers.__version__ == '0.23.2', 'Run the installation cell in a fresh session'
assert torch.cuda.is_available(), 'Включи GPU в настройках Colab'
assert not RESULTS_FILE.exists() and not LATEST_CHECKPOINT.exists(), 'Сначала сохрани прежний запуск; используй новую сессию'
RUN_DIR.mkdir(exist_ok=True)
device = torch.device('cuda')
random.seed(SEED)
torch.manual_seed(SEED)

def sync():
    if device.type == 'cuda': torch.cuda.synchronize()
def sha(raw): return hashlib.sha256(raw).hexdigest()
def data_hash(rows):
    return sha(json.dumps(rows, ensure_ascii=False, sort_keys=True).encode())
def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)
def save_results(): write_json(RESULTS_FILE, result)

result = {'experiment': '10-cosine-width256-bpe4k-20k-pairs', 'phase': 'setup', 'actual_steps': 0,
    'runtime': {'device': str(device), 'gpu': torch.cuda.get_device_name(0),
                'torch': torch.__version__, 'tokenizers': tokenizers.__version__},
    'training_log': [], 'diagnostics': {}, 'checkpoint_files': {},
    'training_seconds_excluding_evaluation_logging_and_checkpoints': 0.0,
    'run_finished': False, 'test_evaluated': False, 'functional_quality_evaluated': False,
    'historical_bpe08': {}}
result['historical_bpe08'] = {'provenance': 'saved user T4 experiment08; not measured in09', 'parameters': 2241024, 'config': {'seed': 42, 'train_size': 20000, 'batch_size_train': 8, 'total_steps': 50000, 'evaluation_steps': [2000, 5000, 10000, 20000, 30000, 50000], 'd_model': 128, 'encoder_layers': 2, 'decoder_layers': 2, 'heads': 4, 'ffn_dim': 512, 'dropout': 0.1, 'activation': 'ReLU', 'norm_first': False, 'lr': 0.001, 'optimizer': 'AdamW', 'weight_decay': 0.01, 'clip_norm': 1.0, 'scheduler': None, 'dtype': 'float32', 'tokenizer': 'own byte-level BPE', 'vocab': 4096, 'content_id_offset': 3, 'special_ids': {'pad': 0, 'bos': 1, 'eos': 2}, 'tokenizer_sha256': '73614c921b43aa6ef715bb23bba583b96785a82304c1964f74ab11d8deef7f86', 'max_positions': 2048, 'generation_cap': 512, 'batch_size_inference': 32, 'inference': 'Post-LN incremental KV cache'}, 'data': {'revision': 'b2724689ff079e9c22efd548fe134bfb29036dbb', 'train_sha256': '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e', 'validation64_sha256': '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050', 'train_row_indices_sha256': '8280c88a7aa6ba3ebe31bfdf9123e5d3fb7a8b3d57272c6fe7fc7e02abdaf142', 'extra_heldout_sha256': 'a2140e9f3e05f8ea10e4647b3a28335d4c9e240c083c14e8b23cf30a6dbe8bfc'}, 'training_seconds': 999.6334417310005, 'diagnostics': {'2000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 2.4418765686915958, 'seconds': 2.1401510529999825, 'answers_per_second': 29.904431236424752, 'generated_tokens_including_eos': 14701, 'generated_utf8_bytes': 38791, 'mean_tokens_including_eos': 229.703125, 'mean_utf8_bytes': 606.109375, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 102.2265625, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 3}, 'parses_as_python_count': 61, 'defines_solve_count': 61, 'solve_returns_value_count': 39, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 41, 'reached_token_cap_count': 23, 'token_cap_without_eos_count': 23, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 6869.141306354379, 'output_bytes_per_second': 18125.35612643988}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 2.632015879918426, 'seconds': 2.1433186589999877, 'answers_per_second': 29.860235542325096, 'generated_tokens_including_eos': 12137, 'generated_utf8_bytes': 34234, 'mean_tokens_including_eos': 189.640625, 'mean_utf8_bytes': 534.90625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 100.876953125, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 9}, 'parses_as_python_count': 55, 'defines_solve_count': 55, 'solve_returns_value_count': 40, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 46, 'reached_token_cap_count': 18, 'token_cap_without_eos_count': 18, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 5662.713730893745, 'output_bytes_per_second': 15972.426618061834}}, '5000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.9814691374367903, 'seconds': 2.102356966000002, 'answers_per_second': 30.442023421820714, 'generated_tokens_including_eos': 4947, 'generated_utf8_bytes': 15308, 'mean_tokens_including_eos': 77.296875, 'mean_utf8_bytes': 239.1875, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 106.306640625, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 2}, 'parses_as_python_count': 62, 'defines_solve_count': 62, 'solve_returns_value_count': 58, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 59, 'reached_token_cap_count': 5, 'token_cap_without_eos_count': 5, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 2353.073279183548, 'output_bytes_per_second': 7281.351477206742}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 2.247825695777351, 'seconds': 2.643643147999967, 'answers_per_second': 24.209016276806814, 'generated_tokens_including_eos': 4495, 'generated_utf8_bytes': 15210, 'mean_tokens_including_eos': 70.234375, 'mean_utf8_bytes': 237.65625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 105.98828125, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 1}, 'parses_as_python_count': 63, 'defines_solve_count': 63, 'solve_returns_value_count': 59, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 60, 'reached_token_cap_count': 4, 'token_cap_without_eos_count': 4, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 1700.3051275663536, 'output_bytes_per_second': 5753.424024534869}}, '10000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.7165860604077952, 'seconds': 2.116977818999999, 'answers_per_second': 30.2317763679885, 'generated_tokens_including_eos': 11691, 'generated_utf8_bytes': 42815, 'mean_tokens_including_eos': 182.671875, 'mean_utf8_bytes': 668.984375, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 106.58984375, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 14}, 'parses_as_python_count': 50, 'defines_solve_count': 50, 'solve_returns_value_count': 44, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 47, 'reached_token_cap_count': 17, 'token_cap_without_eos_count': 17, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 5522.495273721149, 'output_bytes_per_second': 20224.586018678558}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 1.9862113264380399, 'seconds': 2.096558356999992, 'answers_per_second': 30.526219213653988, 'generated_tokens_including_eos': 8869, 'generated_utf8_bytes': 34905, 'mean_tokens_including_eos': 138.578125, 'mean_utf8_bytes': 545.390625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 104.365234375, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 8}, 'parses_as_python_count': 56, 'defines_solve_count': 56, 'solve_returns_value_count': 51, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 52, 'reached_token_cap_count': 12, 'token_cap_without_eos_count': 12, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 4230.266221967144, 'output_bytes_per_second': 16648.713775821758}}, '20000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.4220159569896647, 'seconds': 2.1761731770000097, 'answers_per_second': 29.409424156320128, 'generated_tokens_including_eos': 12248, 'generated_utf8_bytes': 36985, 'mean_tokens_including_eos': 191.375, 'mean_utf8_bytes': 577.890625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 104.7294921875, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 5}, 'parses_as_python_count': 59, 'defines_solve_count': 59, 'solve_returns_value_count': 43, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 45, 'reached_token_cap_count': 19, 'token_cap_without_eos_count': 19, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 5628.2285479157645, 'output_bytes_per_second': 16995.430506585937}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 1.7573962367198745, 'seconds': 2.1413939690000916, 'answers_per_second': 29.887073993154253, 'generated_tokens_including_eos': 11910, 'generated_utf8_bytes': 37841, 'mean_tokens_including_eos': 186.09375, 'mean_utf8_bytes': 591.265625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 103.2392578125, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 4}, 'parses_as_python_count': 60, 'defines_solve_count': 60, 'solve_returns_value_count': 47, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 47, 'reached_token_cap_count': 17, 'token_cap_without_eos_count': 17, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 5561.797675913549, 'output_bytes_per_second': 17671.199483983593}}, '30000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.245738484094068, 'seconds': 2.108978314000069, 'answers_per_second': 30.34644764962619, 'generated_tokens_including_eos': 10284, 'generated_utf8_bytes': 33113, 'mean_tokens_including_eos': 160.6875, 'mean_utf8_bytes': 517.390625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 109.515625, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 4, 'IndentationError': 1}, 'parses_as_python_count': 59, 'defines_solve_count': 59, 'solve_returns_value_count': 47, 'indentation_error_count': 1, 'valid_utf8_count': 64, 'eos_emitted_count': 50, 'reached_token_cap_count': 14, 'token_cap_without_eos_count': 14, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 4876.2948066993085, 'output_bytes_per_second': 15700.967515969875}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 1.6654108966380758, 'seconds': 2.1025492779999695, 'answers_per_second': 30.439239008409547, 'generated_tokens_including_eos': 8105, 'generated_utf8_bytes': 28137, 'mean_tokens_including_eos': 126.640625, 'mean_utf8_bytes': 439.640625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 107.712890625, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 5}, 'parses_as_python_count': 59, 'defines_solve_count': 59, 'solve_returns_value_count': 52, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 55, 'reached_token_cap_count': 9, 'token_cap_without_eos_count': 9, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3854.844252549365, 'output_bytes_per_second': 13382.326062181553}}, '50000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.070016666481843, 'seconds': 2.091999248999855, 'answers_per_second': 30.592745208009603, 'generated_tokens_including_eos': 7434, 'generated_utf8_bytes': 25931, 'mean_tokens_including_eos': 116.15625, 'mean_utf8_bytes': 405.171875, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 107.38818359375, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 3}, 'parses_as_python_count': 61, 'defines_solve_count': 61, 'solve_returns_value_count': 55, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 55, 'reached_token_cap_count': 9, 'token_cap_without_eos_count': 9, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3553.5385605678657, 'output_bytes_per_second': 12395.319937326516}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 1.5732825163916297, 'seconds': 2.0869921720000093, 'answers_per_second': 30.666142814837407, 'generated_tokens_including_eos': 8251, 'generated_utf8_bytes': 26118, 'mean_tokens_including_eos': 128.921875, 'mean_utf8_bytes': 408.09375, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 105.46044921875, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 5}, 'parses_as_python_count': 59, 'defines_solve_count': 59, 'solve_returns_value_count': 53, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 54, 'reached_token_cap_count': 10, 'token_cap_without_eos_count': 10, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3953.5366307066165, 'output_bytes_per_second': 12514.661219342554}}}, 'final_validation': {'validation192_extra': {'split': 'validation192_extra', 'examples': 192, 'bpe_loss': 1.5744023576752602, 'seconds': 6.970978190000096, 'answers_per_second': 27.54276297628143, 'generated_tokens_including_eos': 22261, 'generated_utf8_bytes': 68776, 'mean_tokens_including_eos': 115.94270833333333, 'mean_utf8_bytes': 358.2083333333333, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 108.62744140625, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 7}, 'parses_as_python_count': 185, 'defines_solve_count': 185, 'solve_returns_value_count': 161, 'indentation_error_count': 0, 'valid_utf8_count': 192, 'eos_emitted_count': 165, 'reached_token_cap_count': 27, 'token_cap_without_eos_count': 27, 'byte_budget_stop_count': 0, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3193.38253445313, 'output_bytes_per_second': 9866.04722112881}, 'validation64_byte512': {'split': 'validation64_byte512', 'examples': 64, 'bpe_loss': 1.5732825163916297, 'seconds': 1.4449191549999796, 'answers_per_second': 44.29313555608646, 'generated_tokens_including_eos': 5127, 'generated_utf8_bytes': 16499, 'mean_tokens_including_eos': 80.109375, 'mean_utf8_bytes': 257.796875, 'max_new_tokens': 512, 'max_output_bytes': 512, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 105.39697265625, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 5}, 'parses_as_python_count': 59, 'defines_solve_count': 59, 'solve_returns_value_count': 53, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 54, 'reached_token_cap_count': 0, 'token_cap_without_eos_count': 0, 'byte_budget_stop_count': 10, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3548.295406188364, 'output_bytes_per_second': 11418.631930310477}}, 'test': {'split': 'test1919', 'examples': 1919, 'bpe_loss': None, 'seconds': 66.91523659100017, 'answers_per_second': 28.678072405681338, 'generated_tokens_including_eos': 226657, 'generated_utf8_bytes': 719975, 'mean_tokens_including_eos': 118.11203751954143, 'mean_utf8_bytes': 375.1823866597186, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 113.26318359375, 'timer_scope': 'encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading', 'syntax_error_types': {'SyntaxError': 90, 'IndentationError': 2}, 'parses_as_python_count': 1827, 'defines_solve_count': 1827, 'solve_returns_value_count': 1621, 'indentation_error_count': 2, 'valid_utf8_count': 1919, 'eos_emitted_count': 1655, 'reached_token_cap_count': 264, 'token_cap_without_eos_count': 264, 'byte_budget_stop_count': 0, 'bpe_tokens_per_second': 3387.2255639679597, 'output_bytes_per_second': 10759.507649963742, 'within_180_seconds': True}, 'arithmetic_ast_summaries': {'2000/train64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'numeric': 18, 'undefined_name': 13, 'unsupported': 29, 'syntax_error': 3, 'no_return': 1}, 'reference_statuses': {'numeric': 59, 'unsupported': 5}, 'both_numeric': 18, 'numeric_matches': 0}, '2000/validation64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'numeric': 16, 'syntax_error': 9, 'unsupported': 30, 'undefined_name': 9}, 'reference_statuses': {'numeric': 53, 'unsupported': 11}, 'both_numeric': 15, 'numeric_matches': 1}, '5000/train64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'numeric': 35, 'syntax_error': 2, 'unsupported': 11, 'undefined_name': 16}, 'reference_statuses': {'numeric': 59, 'unsupported': 5}, 'both_numeric': 31, 'numeric_matches': 0}, '5000/validation64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'undefined_name': 26, 'numeric': 27, 'unsupported': 10, 'syntax_error': 1}, 'reference_statuses': {'numeric': 53, 'unsupported': 11}, 'both_numeric': 22, 'numeric_matches': 0}, '10000/train64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'undefined_name': 13, 'syntax_error': 14, 'numeric': 19, 'unsupported': 18}, 'reference_statuses': {'numeric': 59, 'unsupported': 5}, 'both_numeric': 16, 'numeric_matches': 1}, '10000/validation64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'syntax_error': 8, 'unsupported': 19, 'undefined_name': 14, 'numeric': 23}, 'reference_statuses': {'numeric': 53, 'unsupported': 11}, 'both_numeric': 19, 'numeric_matches': 0}, '20000/train64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'undefined_name': 29, 'numeric': 5, 'unsupported': 25, 'syntax_error': 5}, 'reference_statuses': {'numeric': 59, 'unsupported': 5}, 'both_numeric': 5, 'numeric_matches': 1}, '20000/validation64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'undefined_name': 14, 'numeric': 21, 'unsupported': 25, 'syntax_error': 4}, 'reference_statuses': {'numeric': 53, 'unsupported': 11}, 'both_numeric': 18, 'numeric_matches': 1}, '30000/train64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'undefined_name': 27, 'unsupported': 14, 'numeric': 17, 'syntax_error': 5, 'arithmetic_error': 1}, 'reference_statuses': {'numeric': 59, 'unsupported': 5}, 'both_numeric': 15, 'numeric_matches': 1}, '30000/validation64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'unsupported': 5, 'numeric': 26, 'undefined_name': 27, 'syntax_error': 5, 'arithmetic_error': 1}, 'reference_statuses': {'numeric': 53, 'unsupported': 11}, 'both_numeric': 22, 'numeric_matches': 1}, '50000/train64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'numeric': 36, 'undefined_name': 24, 'arithmetic_error': 1, 'syntax_error': 3}, 'reference_statuses': {'numeric': 59, 'unsupported': 5}, 'both_numeric': 34, 'numeric_matches': 4}, '50000/validation64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'undefined_name': 28, 'unsupported': 3, 'numeric': 27, 'syntax_error': 5, 'arithmetic_error': 1}, 'reference_statuses': {'numeric': 53, 'unsupported': 11}, 'both_numeric': 24, 'numeric_matches': 3}, 'validation192_extra': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'undefined_name': 61, 'numeric': 111, 'unsupported': 12, 'syntax_error': 7, 'arithmetic_error': 1}, 'reference_statuses': {'numeric': 170, 'unsupported': 22}, 'both_numeric': 97, 'numeric_matches': 10}, 'historical_byte_50000/validation64': {'scope': 'restricted straight-line numeric AST; no exec/eval, calls, imports, loops, conditions or strings; unsupported cases not scored; reference may be wrong; not pass@1', 'generated_statuses': {'numeric': 21, 'undefined_name': 19, 'syntax_error': 23, 'unsupported': 1}, 'reference_statuses': {'numeric': 53, 'unsupported': 11}, 'both_numeric': 19, 'numeric_matches': 2}}, 'extra_diagnostics': {'2000/train64': {'numeric_ast_comparable_count': 18, 'numeric_ast_match_count': 0, 'numeric_ast_undefined_name_count': 13, 'numeric_ast_statuses': {'numeric': 18, 'undefined_name': 13, 'unsupported': 29, 'syntax_error': 3, 'no_return': 1}, 'repeated_tail_32_plus_count': 23, 'consecutive_lines_3_plus_count': 9}, '2000/validation64': {'numeric_ast_comparable_count': 15, 'numeric_ast_match_count': 1, 'numeric_ast_undefined_name_count': 9, 'numeric_ast_statuses': {'numeric': 16, 'syntax_error': 9, 'unsupported': 30, 'undefined_name': 9}, 'repeated_tail_32_plus_count': 18, 'consecutive_lines_3_plus_count': 13}, '5000/train64': {'numeric_ast_comparable_count': 31, 'numeric_ast_match_count': 0, 'numeric_ast_undefined_name_count': 16, 'numeric_ast_statuses': {'numeric': 35, 'syntax_error': 2, 'unsupported': 11, 'undefined_name': 16}, 'repeated_tail_32_plus_count': 3, 'consecutive_lines_3_plus_count': 1}, '5000/validation64': {'numeric_ast_comparable_count': 22, 'numeric_ast_match_count': 0, 'numeric_ast_undefined_name_count': 26, 'numeric_ast_statuses': {'undefined_name': 26, 'numeric': 27, 'unsupported': 10, 'syntax_error': 1}, 'repeated_tail_32_plus_count': 4, 'consecutive_lines_3_plus_count': 0}, '10000/train64': {'numeric_ast_comparable_count': 16, 'numeric_ast_match_count': 1, 'numeric_ast_undefined_name_count': 13, 'numeric_ast_statuses': {'undefined_name': 13, 'syntax_error': 14, 'numeric': 19, 'unsupported': 18}, 'repeated_tail_32_plus_count': 8, 'consecutive_lines_3_plus_count': 20}, '10000/validation64': {'numeric_ast_comparable_count': 19, 'numeric_ast_match_count': 0, 'numeric_ast_undefined_name_count': 14, 'numeric_ast_statuses': {'syntax_error': 8, 'unsupported': 19, 'undefined_name': 14, 'numeric': 23}, 'repeated_tail_32_plus_count': 2, 'consecutive_lines_3_plus_count': 18}, '20000/train64': {'numeric_ast_comparable_count': 5, 'numeric_ast_match_count': 1, 'numeric_ast_undefined_name_count': 29, 'numeric_ast_statuses': {'undefined_name': 29, 'numeric': 5, 'unsupported': 25, 'syntax_error': 5}, 'repeated_tail_32_plus_count': 13, 'consecutive_lines_3_plus_count': 9}, '20000/validation64': {'numeric_ast_comparable_count': 18, 'numeric_ast_match_count': 1, 'numeric_ast_undefined_name_count': 14, 'numeric_ast_statuses': {'undefined_name': 14, 'numeric': 21, 'unsupported': 25, 'syntax_error': 4}, 'repeated_tail_32_plus_count': 11, 'consecutive_lines_3_plus_count': 8}, '30000/train64': {'numeric_ast_comparable_count': 15, 'numeric_ast_match_count': 1, 'numeric_ast_undefined_name_count': 27, 'numeric_ast_statuses': {'undefined_name': 27, 'unsupported': 14, 'numeric': 17, 'syntax_error': 5, 'arithmetic_error': 1}, 'repeated_tail_32_plus_count': 9, 'consecutive_lines_3_plus_count': 3}, '30000/validation64': {'numeric_ast_comparable_count': 22, 'numeric_ast_match_count': 1, 'numeric_ast_undefined_name_count': 27, 'numeric_ast_statuses': {'unsupported': 5, 'numeric': 26, 'undefined_name': 27, 'syntax_error': 5, 'arithmetic_error': 1}, 'repeated_tail_32_plus_count': 7, 'consecutive_lines_3_plus_count': 10}, '50000/train64': {'numeric_ast_comparable_count': 34, 'numeric_ast_match_count': 4, 'numeric_ast_undefined_name_count': 24, 'numeric_ast_statuses': {'numeric': 36, 'undefined_name': 24, 'arithmetic_error': 1, 'syntax_error': 3}, 'repeated_tail_32_plus_count': 6, 'consecutive_lines_3_plus_count': 7}, '50000/validation64': {'numeric_ast_comparable_count': 24, 'numeric_ast_match_count': 3, 'numeric_ast_undefined_name_count': 28, 'numeric_ast_statuses': {'undefined_name': 28, 'unsupported': 3, 'numeric': 27, 'syntax_error': 5, 'arithmetic_error': 1}, 'repeated_tail_32_plus_count': 7, 'consecutive_lines_3_plus_count': 6}, 'validation192_extra': {'numeric_ast_comparable_count': 97, 'numeric_ast_match_count': 10, 'numeric_ast_undefined_name_count': 61, 'numeric_ast_statuses': {'undefined_name': 61, 'numeric': 111, 'unsupported': 12, 'syntax_error': 7, 'arithmetic_error': 1}, 'repeated_tail_32_plus_count': 19, 'consecutive_lines_3_plus_count': 14}, 'validation64_byte512': {'numeric_ast_comparable_count': 24, 'numeric_ast_match_count': 3, 'numeric_ast_undefined_name_count': 28, 'numeric_ast_statuses': {'undefined_name': 28, 'unsupported': 3, 'numeric': 27, 'syntax_error': 5, 'arithmetic_error': 1}, 'repeated_tail_32_plus_count': 6, 'consecutive_lines_3_plus_count': 6}}}
result['historical_wide09'] = {'provenance': 'saved user T4 run09 plus CPU/static analysis; not measured in10', 'parameters': 6312960, 'config': {'seed': 42, 'train_size': 20000, 'batch_size_train': 8, 'total_steps': 50000, 'evaluation_steps': [2000, 5000, 10000, 20000, 30000, 50000], 'd_model': 256, 'encoder_layers': 2, 'decoder_layers': 2, 'heads': 4, 'ffn_dim': 1024, 'dropout': 0.1, 'activation': 'ReLU', 'norm_first': False, 'lr': 0.001, 'optimizer': 'AdamW', 'weight_decay': 0.01, 'clip_norm': 1.0, 'scheduler': None, 'dtype': 'float32', 'tokenizer': 'own byte-level BPE', 'vocab': 4096, 'content_id_offset': 3, 'special_ids': {'pad': 0, 'bos': 1, 'eos': 2}, 'tokenizer_sha256': '73614c921b43aa6ef715bb23bba583b96785a82304c1964f74ab11d8deef7f86', 'max_positions': 2048, 'generation_cap': 512, 'batch_size_inference': 32, 'inference': 'Post-LN incremental KV cache', 'tokenizer_source_experiment': '08'}, 'data': {'revision': 'b2724689ff079e9c22efd548fe134bfb29036dbb', 'train_sha256': '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e', 'validation64_sha256': '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050', 'train_row_indices_sha256': '8280c88a7aa6ba3ebe31bfdf9123e5d3fb7a8b3d57272c6fe7fc7e02abdaf142', 'extra_heldout_sha256': 'a2140e9f3e05f8ea10e4647b3a28335d4c9e240c083c14e8b23cf30a6dbe8bfc'}, 'diagnostics': {'2000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 2.379917524710936, 'seconds': 2.2088064710000026, 'answers_per_second': 28.974924168446954, 'generated_tokens_including_eos': 13129, 'generated_utf8_bytes': 49071, 'mean_tokens_including_eos': 205.140625, 'mean_utf8_bytes': 766.734375, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 205.8203125, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.027934856000001673, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'numeric': 41, 'unsupported': 21, 'undefined_name': 2}, 'numeric_ast_undefined_name_count': 2, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 4, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 41, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 41, 'reached_token_cap_count': 23, 'token_cap_without_eos_count': 23, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 38, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 5943.934053242813, 'output_bytes_per_second': 22216.07037296657}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 2.541556096580344, 'seconds': 2.16570709399997, 'answers_per_second': 29.551549319531798, 'generated_tokens_including_eos': 10255, 'generated_utf8_bytes': 38210, 'mean_tokens_including_eos': 160.234375, 'mean_utf8_bytes': 597.03125, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 202.689453125, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.024579668999990645, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'numeric': 47, 'unsupported': 16, 'undefined_name': 1}, 'numeric_ast_undefined_name_count': 1, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 7, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 47, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 47, 'reached_token_cap_count': 17, 'token_cap_without_eos_count': 17, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 40, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 4735.174035496853, 'output_bytes_per_second': 17643.19842967672}}, '5000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.9550000658434616, 'seconds': 2.192852635999998, 'answers_per_second': 29.18572773624359, 'generated_tokens_including_eos': 22117, 'generated_utf8_bytes': 87860, 'mean_tokens_including_eos': 345.578125, 'mean_utf8_bytes': 1372.8125, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 208.142578125, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.04558482599998115, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'unsupported': 61, 'numeric': 3}, 'numeric_ast_undefined_name_count': 0, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 36, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 23, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 23, 'reached_token_cap_count': 41, 'token_cap_without_eos_count': 41, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 3, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 10085.949067851554, 'output_bytes_per_second': 40066.5318579119}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 2.205117897429073, 'seconds': 2.3424829059999865, 'answers_per_second': 27.32143736719348, 'generated_tokens_including_eos': 24390, 'generated_utf8_bytes': 97057, 'mean_tokens_including_eos': 381.09375, 'mean_utf8_bytes': 1516.515625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 205.01171875, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.09479385899999215, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'unsupported': 59, 'numeric': 5}, 'numeric_ast_undefined_name_count': 0, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 36, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 18, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 18, 'reached_token_cap_count': 46, 'token_cap_without_eos_count': 46, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 5, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 10412.029021653889, 'output_bytes_per_second': 41433.386664807775}}, '10000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.766058729364428, 'seconds': 0.2869437960000596, 'answers_per_second': 223.04019425458046, 'generated_tokens_including_eos': 3576, 'generated_utf8_bytes': 13223, 'mean_tokens_including_eos': 55.875, 'mean_utf8_bytes': 206.609375, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 209.0341796875, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.018613675999972656, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 64}, 'numeric_ast_undefined_name_count': 64, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 0, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 64, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 64, 'reached_token_cap_count': 0, 'token_cap_without_eos_count': 0, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 0, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 12462.370853974684, 'output_bytes_per_second': 46082.19513481746}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 2.0498782869950367, 'seconds': 0.2991647119999925, 'answers_per_second': 213.92897435042943, 'generated_tokens_including_eos': 3528, 'generated_utf8_bytes': 13313, 'mean_tokens_including_eos': 55.125, 'mean_utf8_bytes': 208.015625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 205.9033203125, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.01857098200002838, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 64}, 'numeric_ast_undefined_name_count': 64, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 0, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 64, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 64, 'reached_token_cap_count': 0, 'token_cap_without_eos_count': 0, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 0, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 11792.834711067422, 'output_bytes_per_second': 44500.569305113546}}, '20000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.6026217711290514, 'seconds': 2.074608487999967, 'answers_per_second': 30.84919413479283, 'generated_tokens_including_eos': 5085, 'generated_utf8_bytes': 14549, 'mean_tokens_including_eos': 79.453125, 'mean_utf8_bytes': 227.328125, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 207.7763671875, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.017593529000009767, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 46, 'numeric': 18}, 'numeric_ast_undefined_name_count': 46, 'repeated_tail_32_plus_count': 5, 'consecutive_lines_3_plus_count': 0, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 59, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 59, 'reached_token_cap_count': 5, 'token_cap_without_eos_count': 5, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 17, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 2451.0648777409615, 'output_bytes_per_second': 7012.88946042345}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 1.943692813221644, 'seconds': 1.1809605419999798, 'answers_per_second': 54.19317388167326, 'generated_tokens_including_eos': 3257, 'generated_utf8_bytes': 11257, 'mean_tokens_including_eos': 50.890625, 'mean_utf8_bytes': 175.890625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 204.6455078125, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.019142452999972193, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 48, 'numeric': 16}, 'numeric_ast_undefined_name_count': 48, 'repeated_tail_32_plus_count': 1, 'consecutive_lines_3_plus_count': 0, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 63, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 63, 'reached_token_cap_count': 1, 'token_cap_without_eos_count': 1, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 12, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 2757.924489572028, 'output_bytes_per_second': 9532.071224781186}}, '30000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.474684315255722, 'seconds': 2.136424868000063, 'answers_per_second': 29.956588204251382, 'generated_tokens_including_eos': 21797, 'generated_utf8_bytes': 73457, 'mean_tokens_including_eos': 340.578125, 'mean_utf8_bytes': 1147.765625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 212.6484375, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.04288157799999226, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'unsupported': 41, 'numeric': 23}, 'numeric_ast_undefined_name_count': 0, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 41, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 23, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 23, 'reached_token_cap_count': 41, 'token_cap_without_eos_count': 41, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 21, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 10202.558642001053, 'output_bytes_per_second': 34383.14218312022}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 1.9181940001855655, 'seconds': 2.4459825890000957, 'answers_per_second': 26.165353869572247, 'generated_tokens_including_eos': 24659, 'generated_utf8_bytes': 83111, 'mean_tokens_including_eos': 385.296875, 'mean_utf8_bytes': 1298.609375, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 209.517578125, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.08787398900005883, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'unsupported': 47, 'numeric': 17}, 'numeric_ast_undefined_name_count': 0, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 47, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 17, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 17, 'reached_token_cap_count': 47, 'token_cap_without_eos_count': 47, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 13, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 10081.429079215344, 'output_bytes_per_second': 33978.573835219046}}, '50000': {'train64': {'split': 'train64', 'examples': 64, 'bpe_loss': 1.382657334684395, 'seconds': 2.118585467999992, 'answers_per_second': 30.20883554932429, 'generated_tokens_including_eos': 11956, 'generated_utf8_bytes': 41780, 'mean_tokens_including_eos': 186.8125, 'mean_utf8_bytes': 652.8125, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 210.73974609375, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.028324451000116824, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 44, 'unsupported': 20}, 'numeric_ast_undefined_name_count': 44, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 0, 'syntax_error_types': {}, 'parses_as_python_count': 64, 'defines_solve_count': 64, 'solve_returns_value_count': 44, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 44, 'reached_token_cap_count': 20, 'token_cap_without_eos_count': 20, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 0, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 59, 'unsupported': 5}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 5643.388091058144, 'output_bytes_per_second': 19720.705457043263}, 'validation64': {'split': 'validation64', 'examples': 64, 'bpe_loss': 1.9436050678626589, 'seconds': 2.116559251000126, 'answers_per_second': 30.237754964694908, 'generated_tokens_including_eos': 8645, 'generated_utf8_bytes': 30586, 'mean_tokens_including_eos': 135.078125, 'mean_utf8_bytes': 477.90625, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 207.67138671875, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.025281632999849535, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 51, 'unsupported': 12, 'syntax_error': 1}, 'numeric_ast_undefined_name_count': 51, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 1, 'syntax_error_types': {'SyntaxError': 1}, 'parses_as_python_count': 63, 'defines_solve_count': 63, 'solve_returns_value_count': 51, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 51, 'reached_token_cap_count': 13, 'token_cap_without_eos_count': 13, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 0, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 4084.4592448404296, 'output_bytes_per_second': 14450.812083596225}}}, 'final_validation': {'validation192_extra': {'split': 'validation192_extra', 'examples': 192, 'bpe_loss': 1.9277490218772468, 'seconds': 7.081739189000018, 'answers_per_second': 27.111984058694414, 'generated_tokens_including_eos': 26881, 'generated_utf8_bytes': 94769, 'mean_tokens_including_eos': 140.00520833333334, 'mean_utf8_bytes': 493.5885416666667, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 214.05712890625, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.07761672099991301, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 151, 'unsupported': 41}, 'numeric_ast_undefined_name_count': 151, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 0, 'syntax_error_types': {}, 'parses_as_python_count': 192, 'defines_solve_count': 192, 'solve_returns_value_count': 151, 'indentation_error_count': 0, 'valid_utf8_count': 192, 'eos_emitted_count': 151, 'reached_token_cap_count': 41, 'token_cap_without_eos_count': 41, 'byte_budget_stop_count': 0, 'numeric_ast_comparable_count': 0, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 170, 'unsupported': 22}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 3795.8189764675235, 'output_bytes_per_second': 13382.16467322089}, 'validation64_byte512': {'split': 'validation64_byte512', 'examples': 64, 'bpe_loss': 1.9436050678626589, 'seconds': 0.7176505029999589, 'answers_per_second': 89.17989987112664, 'generated_tokens_including_eos': 3932, 'generated_utf8_bytes': 14305, 'mean_tokens_including_eos': 61.4375, 'mean_utf8_bytes': 223.515625, 'max_new_tokens': 512, 'max_output_bytes': 512, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 207.5830078125, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.01837766200014812, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'undefined_name': 51, 'unsupported': 12, 'syntax_error': 1}, 'numeric_ast_undefined_name_count': 51, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 1, 'syntax_error_types': {'SyntaxError': 1}, 'parses_as_python_count': 63, 'defines_solve_count': 63, 'solve_returns_value_count': 51, 'indentation_error_count': 0, 'valid_utf8_count': 64, 'eos_emitted_count': 51, 'reached_token_cap_count': 0, 'token_cap_without_eos_count': 0, 'byte_budget_stop_count': 13, 'numeric_ast_comparable_count': 0, 'numeric_ast_match_count': 0, 'reference_numeric_ast_statuses': {'numeric': 53, 'unsupported': 11}, 'exact_code_match_count': 0, 'exact_tokens_with_eos_count': 0, 'bpe_tokens_per_second': 5478.990098332342, 'output_bytes_per_second': 19933.101057132288}}, 'test': {'split': 'test1919', 'examples': 1919, 'bpe_loss': None, 'seconds': 68.35823648600012, 'answers_per_second': 28.072696117504645, 'generated_tokens_including_eos': 329315, 'generated_utf8_bytes': 1153918, 'mean_tokens_including_eos': 171.60760812923397, 'mean_utf8_bytes': 601.3121417404899, 'max_new_tokens': 512, 'max_output_bytes': None, 'batch_size_inference': 32, 'peak_gpu_memory_mib': 222.28955078125, 'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics', 'extra_diagnostics_seconds_excluded': 0.5867058279998218, 'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1', 'numeric_ast_statuses': {'unsupported': 532, 'undefined_name': 1381, 'syntax_error': 6}, 'numeric_ast_undefined_name_count': 1381, 'repeated_tail_32_plus_count': 0, 'consecutive_lines_3_plus_count': 0, 'syntax_error_types': {'SyntaxError': 6}, 'parses_as_python_count': 1913, 'defines_solve_count': 1913, 'solve_returns_value_count': 1381, 'indentation_error_count': 0, 'valid_utf8_count': 1919, 'eos_emitted_count': 1381, 'reached_token_cap_count': 538, 'token_cap_without_eos_count': 538, 'byte_budget_stop_count': 0, 'bpe_tokens_per_second': 4817.488234463805, 'output_bytes_per_second': 16880.45302684665, 'within_180_seconds': True}, 'training_seconds': 1004.0231621879987, 'diversity': {'2000/train64': {'unique_generated_code_count': 24, 'most_common_generated_code_count': 41, 'most_common_fraction': 0.640625, 'most_common_code': 'def solve() -> int:\n    length = 8\n    width = 8\n    perimeter = 2\n    perimeter = width * width\n    result = perimeter\n\n    return result\n'}, '2000/validation64': {'unique_generated_code_count': 18, 'most_common_generated_code_count': 47, 'most_common_fraction': 0.734375, 'most_common_code': 'def solve() -> int:\n    length = 8\n    width = 8\n    perimeter = 2\n    perimeter = width * width\n    result = perimeter\n\n    return result\n'}, '5000/train64': {'unique_generated_code_count': 5, 'most_common_generated_code_count': 24, 'most_common_fraction': 0.375, 'most_common_code': 'def solve() -> int:\n    total_students = 200\n    boy_students = 0\n    boy_students * boy_students * boy_students - boy_students\n    boy_students\n    boy_students = boy_students\n    boy_students = boy_students - boy_students\n    boy_students\n    boy_students = boy_students\n    boy_students = boy_students - boy_students\n    boy_students\n    boy_students = boy_students\n    boy_students\n    boy_students = boy_students - boy_students\n    boy_students\n    boy_students = boy_students\n    boy_students ='}, '5000/validation64': {'unique_generated_code_count': 7, 'most_common_generated_code_count': 26, 'most_common_fraction': 0.40625, 'most_common_code': 'def solve() -> int:\n    total_students = 200\n    boy_students = 0\n    boy_students * boy_students * boy_students - boy_students\n    boy_students\n    boy_students = boy_students\n    boy_students = boy_students - boy_students\n    boy_students\n    boy_students = boy_students\n    boy_students = boy_students - boy_students\n    boy_students\n    boy_students = boy_students\n    boy_students\n    boy_students = boy_students - boy_students\n    boy_students\n    boy_students = boy_students\n    boy_students ='}, '10000/train64': {'unique_generated_code_count': 2, 'most_common_generated_code_count': 41, 'most_common_fraction': 0.640625, 'most_common_code': 'def solve() -> int:\n    total_students = 60\n    girls = total_students * 0.2\n    boys_students = total_students\n    girls_students = total_students - girls_students\n    result = girls_students\n\n    return result\n'}, '10000/validation64': {'unique_generated_code_count': 2, 'most_common_generated_code_count': 47, 'most_common_fraction': 0.734375, 'most_common_code': 'def solve() -> int:\n    total_students = 60\n    girls = total_students * 0.2\n    boys_students = total_students\n    girls_students = total_students - girls_students\n    result = girls_students\n\n    return result\n'}, '20000/train64': {'unique_generated_code_count': 4, 'most_common_generated_code_count': 41, 'most_common_fraction': 0.640625, 'most_common_code': 'def solve() -> int:\n    total_students = 30\n    girls_students = total_students * 0.6\n    boys_students = total_students - boys_students\n    result = boys_students\n    return result\n'}, '20000/validation64': {'unique_generated_code_count': 4, 'most_common_generated_code_count': 47, 'most_common_fraction': 0.734375, 'most_common_code': 'def solve() -> int:\n    total_students = 30\n    girls_students = total_students * 0.6\n    boys_students = total_students - boys_students\n    result = boys_students\n    return result\n'}, '30000/train64': {'unique_generated_code_count': 3, 'most_common_generated_code_count': 40, 'most_common_fraction': 0.625, 'most_common_code': 'def solve() -> int:\n    total_students = 24\n    boy_ratio = 3/3\n    girl_ratio = boy_ratio * boy_ratio\n    girl_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio * boy_ratio\n    girl_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = girl_ratio\n    girl_ratio = girl_ratio\n    girl_ratio = girl_ratio\n    girl_ratio = girl_ratio / boy_ratio\n    girl_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio '}, '30000/validation64': {'unique_generated_code_count': 4, 'most_common_generated_code_count': 44, 'most_common_fraction': 0.6875, 'most_common_code': 'def solve() -> int:\n    total_students = 24\n    boy_ratio = 3/3\n    girl_ratio = boy_ratio * boy_ratio\n    girl_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio * boy_ratio\n    girl_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = girl_ratio\n    girl_ratio = girl_ratio\n    girl_ratio = girl_ratio\n    girl_ratio = girl_ratio / boy_ratio\n    girl_ratio\n    girl_ratio = boy_ratio\n    girl_ratio = boy_ratio '}, '50000/train64': {'unique_generated_code_count': 9, 'most_common_generated_code_count': 44, 'most_common_fraction': 0.6875, 'most_common_code': 'def solve() -> int:\n    total_students = 30\n    girls = total_students * 0.35\n    girls = total_students - boys\n    result = girls\n\n    return result\n'}, '50000/validation64': {'unique_generated_code_count': 8, 'most_common_generated_code_count': 51, 'most_common_fraction': 0.796875, 'most_common_code': 'def solve() -> int:\n    total_students = 30\n    girls = total_students * 0.35\n    girls = total_students - boys\n    result = girls\n\n    return result\n'}, 'validation192_extra': {'unique_generated_code_count': 9, 'most_common_generated_code_count': 151, 'most_common_fraction': 0.7864583333333334, 'most_common_code': 'def solve() -> int:\n    total_students = 30\n    girls = total_students * 0.35\n    girls = total_students - boys\n    result = girls\n\n    return result\n'}, 'validation64_byte512': {'unique_generated_code_count': 4, 'most_common_generated_code_count': 51, 'most_common_fraction': 0.796875, 'most_common_code': 'def solve() -> int:\n    total_students = 30\n    girls = total_students * 0.35\n    girls = total_students - boys\n    result = girls\n\n    return result\n'}, 'test': {'unique_generated_code_count': 31, 'most_common_generated_code_count': 1381, 'most_common_fraction': 0.7196456487754038, 'most_common_code': 'def solve() -> int:\n    total_students = 30\n    girls = total_students * 0.35\n    girls = total_students - boys\n    result = girls\n\n    return result\n'}}}
save_results()
print('RUNTIME', json.dumps(result['runtime']))


RUNTIME {"device": "cuda", "gpu": "Tesla T4", "torch": "2.11.0+cu130", "tokenizers": "0.23.2"}


## 2. Прежние 20 000 train-пар и контрольные вопросы


In [3]:
revision = 'b2724689ff079e9c22efd548fe134bfb29036dbb'
base_url = f'https://huggingface.co/datasets/SlavaYus/tinysms_qa_12m/resolve/{revision}'
cache = Path('hw02-data')
cache.mkdir(exist_ok=True)
train_file = cache / 'train.parquet'
data_started = time.perf_counter()
if not train_file.exists(): urlretrieve(f'{base_url}/data/train/part-00000.parquet', train_file)
def is_validation(question):
    key = hashlib.sha256(f'{SEED}:{question}'.encode()).digest()
    return int.from_bytes(key[:8], 'big') % 10 == 0
def eligible(row):
    return len(row['question'].encode()) <= 512 and len(row['code'].encode()) < 512

train, train_row_indices, validation = [], [], None
heldout_pool, heldout_questions = [], set()
offset = 0
for batch in pq.ParquetFile(train_file).iter_batches(batch_size=4096):
    rows = batch.to_pylist()
    if validation is None:
        validation = [r for r in rows if eligible(r) and is_validation(r['question'])][:64]
        heldout_questions.update(r['question'] for r in validation)
    for index, row in enumerate(rows):
        if not eligible(row): continue
        if is_validation(row['question']):
            if row['question'] not in heldout_questions and len(heldout_pool) < 192:
                heldout_pool.append(row)
                heldout_questions.add(row['question'])
            continue
        train.append(row)
        train_row_indices.append(offset + index)
        if len(train) == TRAIN_SIZE: break
    if len(train) == TRAIN_SIZE: break
    offset += len(rows)
assert len(train) == 20000 and len(validation) == 64 and len(heldout_pool) == 192
train_hash, validation_hash = data_hash(train), data_hash(validation)
assert train_hash == '1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e'
assert data_hash(train[:512]) == '3801627c7dcaae0c82c6694e0aa8514ca8ced044c07c7f30e779bb6a3130bc35'
assert validation_hash == '0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050'
assert not ({r['question'] for r in train} & heldout_questions)
assert all(not is_validation(r['question']) for r in train)
train_diagnostic = train[:64]
validation_extra = heldout_pool
result['data'] = {'revision': revision, 'shard': 0, 'training_examples': len(train),
    'unique_train_questions': len({r['question'] for r in train}),
    'train_sha256': train_hash, 'validation64_sha256': validation_hash,
    'train_row_indices_sha256': data_hash(train_row_indices),
    'extra_heldout_examples': len(validation_extra), 'extra_heldout_sha256': data_hash(validation_extra),
    'extra_selection': 'first 192 distinct eligible hash-heldout questions after excluding original validation64',
    'question_overlap_train_heldout': 0, 'download_and_selection_seconds': time.perf_counter() - data_started,
    'selection': 'unchanged first20000 non-heldout rows of shard0; question<=512/code<512 UTF8 bytes'}
result['phase'] = 'data_ready'
save_results()
print('DATA', json.dumps(result['data']))


DATA {"revision": "b2724689ff079e9c22efd548fe134bfb29036dbb", "shard": 0, "training_examples": 20000, "unique_train_questions": 19960, "train_sha256": "1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e", "validation64_sha256": "0edf2b479ebdc103ec05efc704c7bb0ac8084e41a8a44b58afef224702563050", "train_row_indices_sha256": "8280c88a7aa6ba3ebe31bfdf9123e5d3fb7a8b3d57272c6fe7fc7e02abdaf142", "extra_heldout_examples": 192, "extra_heldout_sha256": "a2140e9f3e05f8ea10e4647b3a28335d4c9e240c083c14e8b23cf30a6dbe8bfc", "extra_selection": "first 192 distinct eligible hash-heldout questions after excluding original validation64", "question_overlap_train_heldout": 0, "download_and_selection_seconds": 1.691323656999998, "selection": "unchanged first20000 non-heldout rows of shard0; question<=512/code<512 UTF8 bytes"}


## 3. Тот же собственный BPE из08

Сжатая сериализация ниже содержит только наш словарь/merges, обученные на прежних20k train-вопросах и коде. Это не веса Transformer.4093 обычных токена+PAD/BOS/EOS0/1/2; offset3. Normalizer отсутствует, пробелы/новые строки/Unicode сохраняются. Длинную строку со словарём менять не нужно.


In [4]:
tokenizer_started = time.perf_counter()
# Fixed own tokenizer from user run08. No model weights or heldout data embedded.
TOKENIZER_ZLIB_BASE64 = (
    '...'
)
tokenizer_raw = zlib.decompress(base64.b64decode(TOKENIZER_ZLIB_BASE64))
EXPECTED_TOKENIZER_SHA256 = '73614c921b43aa6ef715bb23bba583b96785a82304c1964f74ab11d8deef7f86'
assert sha(tokenizer_raw) == EXPECTED_TOKENIZER_SHA256
TOKENIZER_FILE.write_bytes(tokenizer_raw)
tokenizer = Tokenizer.from_str(tokenizer_raw.decode('utf-8'))
VOCAB = tokenizer.get_vocab_size() + TOKEN_OFFSET
assert VOCAB == TARGET_VOCAB
tokenizer_hash = sha(TOKENIZER_FILE.read_bytes())

# The reversible ByteLevel alphabet, as in tokenizers/src/pre_tokenizers/byte_level.rs.
visible = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256))
byte_chars = {b: chr(b) for b in visible}
for index, b in enumerate(b for b in range(256) if b not in byte_chars):
    byte_chars[b] = chr(256 + index)
char_bytes = {char: b for b, char in byte_chars.items()}
assert set(char_bytes) == set(pre_tokenizers.ByteLevel.alphabet())
TOKEN_BYTES = [b'', b'', b''] + [bytes(char_bytes[c] for c in tokenizer.id_to_token(i))
                                for i in range(VOCAB - TOKEN_OFFSET)]
assert all(TOKEN_BYTES[i] for i in range(TOKEN_OFFSET, VOCAB))

def encode(text): return [i + TOKEN_OFFSET for i in tokenizer.encode(text, add_special_tokens=False).ids]
def decode_bytes(ids):
    parts = []
    for i in ids:
        if i == EOS: break
        if i >= TOKEN_OFFSET: parts.append(TOKEN_BYTES[i])
    return b''.join(parts)
def decode(ids): return decode_bytes(ids).decode('utf-8', errors='replace')
def length_stats(values):
    values = sorted(values)
    return {'count': len(values), 'mean': sum(values) / len(values), 'min': values[0],
        'p50': values[int(0.5 * (len(values) - 1))], 'p95': values[int(0.95 * (len(values) - 1))],
        'max': values[-1]}
def audit_tokenization(rows):
    report = {}
    for field in ('question', 'code'):
        byte_lengths, bpe_lengths = [], []
        for row in rows:
            text = row[field]
            ids = encode(text)
            assert decode_bytes(ids) == text.encode('utf-8'), 'Round-trip failed'
            assert tokenizer.decode([i - TOKEN_OFFSET for i in ids], skip_special_tokens=False) == text
            assert all(TOKEN_OFFSET <= i < VOCAB for i in ids)
            assert len(ids) + 2 <= MAX_POSITIONS
            byte_lengths.append(len(text.encode('utf-8')))
            bpe_lengths.append(len(ids))
        report[field] = {'utf8_bytes': length_stats(byte_lengths), 'bpe_tokens': length_stats(bpe_lengths),
            'total_bytes_per_bpe_token': sum(byte_lengths) / sum(bpe_lengths)}
    return report
probes = ['', '    return 1\n', '\n\n\t x\r\n', '  trailing  ',
          'Привет 世界 🧠', 'x = 0.005 + 123456789\n', '<pad><bos><eos>',
          ''.join(chr(i) for i in range(256))]
for text in probes: assert decode_bytes(encode(text)) == text.encode('utf-8')
reloaded_tokenizer = Tokenizer.from_file(str(TOKENIZER_FILE))
assert all(reloaded_tokenizer.encode(t).ids == tokenizer.encode(t).ids for t in probes)
result['tokenization'] = {'kind': 'own byte-level BPE', 'loaded_from_experiment': '08', 'tokenizer_training_performed': False, 'target_model_vocabulary': TARGET_VOCAB,
    'actual_model_vocabulary': VOCAB, 'content_id_offset': TOKEN_OFFSET, 'special_ids': {'pad': PAD, 'bos': BOS, 'eos': EOS},
    'normalizer': None, 'add_prefix_space': False, 'use_regex': True, 'min_frequency': 2,
    'training_corpus': 'only original20000 train questions and codes; no heldout/test',
    'training_corpus_sha256': train_hash, 'tokenizer_sha256': tokenizer_hash,
    'train_round_trip': audit_tokenization(train), 'validation64_round_trip': audit_tokenization(validation),
    'extra192_round_trip': audit_tokenization(validation_extra), 'probe_round_trip_count': len(probes),
    'loading_and_audit_seconds': time.perf_counter() - tokenizer_started}
result['phase'] = 'tokenizer_ready'
save_results()
print('TOKENIZATION', json.dumps(result['tokenization']))
for text in ['def solve():\n    return 1\n', '    ', 'return', '12345']:
    print('TOKENS', repr(text), [(i, repr(TOKEN_BYTES[i])) for i in encode(text)])

def source_batch(questions):
    return pad_sequence([torch.tensor([BOS] + encode(q) + [EOS], dtype=torch.long) for q in questions],
                        batch_first=True, padding_value=PAD).to(device)
def training_batch(examples):
    src = source_batch([r['question'] for r in examples])
    codes = [encode(r['code']) for r in examples]
    tgt = pad_sequence([torch.tensor([BOS] + c, dtype=torch.long) for c in codes],
                       batch_first=True, padding_value=PAD).to(device)
    labels = pad_sequence([torch.tensor(c + [EOS], dtype=torch.long) for c in codes],
                          batch_first=True, padding_value=PAD).to(device)
    return src, tgt, labels


TOKENIZATION {"kind": "own byte-level BPE", "loaded_from_experiment": "08", "tokenizer_training_performed": false, "target_model_vocabulary": 4096, "actual_model_vocabulary": 4096, "content_id_offset": 3, "special_ids": {"pad": 0, "bos": 1, "eos": 2}, "normalizer": null, "add_prefix_space": false, "use_regex": true, "min_frequency": 2, "training_corpus": "only original20000 train questions and codes; no heldout/test", "training_corpus_sha256": "1304183967f02f01097d76c12fb24df30916a4ec44e54833cd3509264359300e", "tokenizer_sha256": "73614c921b43aa6ef715bb23bba583b96785a82304c1964f74ab11d8deef7f86", "train_round_trip": {"question": {"utf8_bytes": {"count": 20000, "mean": 174.5776, "min": 11, "p50": 160, "p95": 311, "max": 512}, "bpe_tokens": {"count": 20000, "mean": 44.43445, "min": 4, "p50": 41, "p95": 81, "max": 153}, "total_bytes_per_bpe_token": 3.928879506779087}, "code": {"utf8_bytes": {"count": 20000, "mean": 224.6011, "min": 54, "p50": 206, "p95": 406, "max": 511}, "bpe_tokens": {"

## 4. Прежняя широкая модель, новое расписание LR

Классы модели и KV-кеша прежние:2+2/256,FFN1024,4 головы,6.313M. Инициализация с seed42 как в09, новые веса/AdamW; обучающий фактор — **расписание**, начальныйLR0.001 сохранён. Нет warmup, рестартов или изменения Pre/Post-LN.

Для $t$ выполненных обновлений scheduler задаёт LR следующего обновления:

$$\eta(t)=\eta_{min}+\frac{\eta_{max}-\eta_{min}}{2}\left(1+\cos\frac{\pi t}{T}\right).$$

$\eta_{max}=0.001$, $\eta_{min}=0.0001$, $T=50000$; первый update использует $\eta(0)=0.001$. После `optimizer.step()` вызывается `scheduler.step()` один раз. Лог `lr_used` — LR завершённого update, `lr_next` — следующего. При $t=10000$ LR≈0.000914; при25000 —0.00055; после50000 —0.0001. Расписание обновляется по optimizer steps, а не эпохам/оценкам. До/после обучения проверяется cached/full prefix и короткийgreedy.


In [5]:
class SimpleTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.tokens = nn.Embedding(VOCAB, D_MODEL, padding_idx=PAD)
        self.positions = nn.Embedding(MAX_POSITIONS, D_MODEL)
        self.transformer = nn.Transformer(d_model=D_MODEL, nhead=4,
            num_encoder_layers=N_LAYERS, num_decoder_layers=N_LAYERS,
            dim_feedforward=4 * D_MODEL, dropout=0.1, batch_first=True)
        self.transformer.encoder.enable_nested_tensor = False
        self.transformer.encoder.use_nested_tensor = False
        self.head = nn.Linear(D_MODEL, VOCAB)
    def embed(self, ids):
        if ids.shape[1] > MAX_POSITIONS:
            raise ValueError('Sequence too long; no silent truncation')
        pos = torch.arange(ids.shape[1], device=ids.device)
        return self.tokens(ids) * math.sqrt(D_MODEL) + self.positions(pos)
    def decode_prefix(self, tgt, memory, source_padding):
        mask = torch.ones(tgt.shape[1], tgt.shape[1], device=tgt.device, dtype=torch.bool).triu(1)
        return self.transformer.decoder(self.embed(tgt), memory, tgt_mask=mask,
            tgt_key_padding_mask=tgt.eq(PAD), memory_key_padding_mask=source_padding)
    def forward(self, src, tgt):
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        return self.head(self.decode_prefix(tgt, memory, src.eq(PAD)))
    @torch.inference_mode()
    def generate(self, src, max_new_tokens=512):
        self.eval()
        memory = self.transformer.encoder(self.embed(src), src_key_padding_mask=src.eq(PAD))
        tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
        done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
        for _ in range(max_new_tokens):
            hidden = self.decode_prefix(tgt, memory, src.eq(PAD))
            logits = self.head(hidden[:, -1])
            logits[:, [PAD, BOS]] = -torch.inf
            token = logits.argmax(-1).masked_fill(done, PAD)
            done |= token.eq(EOS)
            tgt = torch.cat([tgt, token[:, None]], dim=1)
            if bool(done.all()):
                break
        return tgt[:, 1:]

model = SimpleTransformer().to(device)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=TOTAL_STEPS, eta_min=MIN_LEARNING_RATE)
TOKEN_BYTE_LENGTHS = torch.tensor([len(b) for b in TOKEN_BYTES], device=device)
config = {'seed': SEED, 'train_size': len(train), 'batch_size_train': BATCH_SIZE, 'total_steps': TOTAL_STEPS,
    'evaluation_steps': list(EVALUATION_STEPS), 'd_model': D_MODEL, 'encoder_layers': N_LAYERS,
    'decoder_layers': N_LAYERS, 'heads': 4, 'ffn_dim': 4 * D_MODEL, 'dropout': 0.1,
    'activation': 'ReLU', 'norm_first': False, 'lr': LEARNING_RATE, 'optimizer': 'AdamW',
    'weight_decay': optimizer.param_groups[0]['weight_decay'], 'clip_norm': 1.0, 'scheduler': {'name': 'CosineAnnealingLR', 'T_max': TOTAL_STEPS,
        'eta_min': MIN_LEARNING_RATE, 'warmup_steps': 0, 'restarts': False, 'step_unit': 'optimizer update'},
    'dtype': 'float32', 'tokenizer': 'own byte-level BPE', 'vocab': VOCAB, 'content_id_offset': TOKEN_OFFSET,
    'special_ids': {'pad': PAD, 'bos': BOS, 'eos': EOS}, 'tokenizer_sha256': tokenizer_hash,
    'max_positions': MAX_POSITIONS, 'generation_cap': GENERATION_CAP,
    'batch_size_inference': INFERENCE_BATCH, 'inference': 'Post-LN incremental KV cache'}
config['tokenizer_source_experiment'] = '08'
result['config'] = config
result['parameters'] = sum(p.numel() for p in model.parameters())
assert result['parameters'] == 6312960
result['phase'] = 'model_ready'
save_results()
print('PARAMETERS', result['parameters'])


PARAMETERS 6312960


In [6]:
class CachedDecoder:
    @torch.inference_mode()
    def __init__(self, model, src, capacity):
        model.eval()
        if not 0 < capacity <= model.positions.num_embeddings:
            raise ValueError('Invalid cache capacity')
        self.model, self.capacity, self.position = model, capacity, 0
        self.batch_size = len(src)
        self.width = model.tokens.embedding_dim
        self.heads = model.transformer.decoder.layers[0].self_attn.num_heads
        self.head_dim = self.width // self.heads
        memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
        # SDPA bool masks use True for allowed keys (opposite to MHA padding masks).
        self.source_allowed = src.ne(PAD)[:, None, None, :]
        self.valid = torch.zeros(len(src), capacity, dtype=torch.bool, device=src.device)
        self.position_vectors = model.positions(torch.arange(capacity, device=src.device))
        self.layers = []
        for layer in model.transformer.decoder.layers:
            if layer.norm_first:
                raise ValueError('This experiment implements the saved Post-LN model')
            for attention in (layer.self_attn, layer.multihead_attn):
                if (not attention._qkv_same_embed_dim or attention.num_heads != self.heads
                        or attention.bias_k is not None or attention.bias_v is not None
                        or attention.add_zero_attn):
                    raise ValueError('Unsupported attention configuration')
            attention = layer.multihead_attn
            projected_kv = F.linear(memory, attention.in_proj_weight[self.width:],
                                    attention.in_proj_bias[self.width:])
            cross_k, cross_v = projected_kv.chunk(2, dim=-1)
            shape = (len(src), self.heads, capacity, self.head_dim)
            self.layers.append({
                'self_k': memory.new_empty(shape), 'self_v': memory.new_empty(shape),
                'cross_k': self.split_heads(cross_k), 'cross_v': self.split_heads(cross_v),
            })

    def split_heads(self, values):
        return values.reshape(self.batch_size, -1, self.heads, self.head_dim).transpose(1, 2)

    def attend(self, attention, q, k, v, allowed):
        # One query at the latest position; all supplied keys are from its past/current.
        # is_causal=True would use an upper-left 1 x T mask and incorrectly hide the past.
        output = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed,
                                                dropout_p=0.0, is_causal=False)
        output = output.transpose(1, 2).contiguous().reshape(self.batch_size, 1, self.width)
        return attention.out_proj(output)

    @torch.inference_mode()
    def step(self, token_ids):
        if self.position >= self.capacity:
            raise ValueError('Cache capacity exceeded')
        if token_ids.ndim != 1 or len(token_ids) != self.batch_size:
            raise ValueError('One token per batch row is required')
        t = self.position
        x = self.model.tokens(token_ids[:, None]) * math.sqrt(self.width)
        x = x + self.position_vectors[t][None, None, :]
        self.valid[:, t] = token_ids.ne(PAD)
        self_allowed = self.valid[:, None, None, :t + 1]
        for layer, cache in zip(self.model.transformer.decoder.layers, self.layers):
            attention = layer.self_attn
            q, k, v = F.linear(x, attention.in_proj_weight, attention.in_proj_bias).chunk(3, dim=-1)
            cache['self_k'][:, :, t:t + 1] = self.split_heads(k)
            cache['self_v'][:, :, t:t + 1] = self.split_heads(v)
            self_output = self.attend(attention, self.split_heads(q),
                cache['self_k'][:, :, :t + 1], cache['self_v'][:, :, :t + 1], self_allowed)
            x = layer.norm1(x + layer.dropout1(self_output))

            attention = layer.multihead_attn
            q = F.linear(x, attention.in_proj_weight[:self.width], attention.in_proj_bias[:self.width])
            cross_output = self.attend(attention, self.split_heads(q),
                                      cache['cross_k'], cache['cross_v'], self.source_allowed)
            x = layer.norm2(x + layer.dropout2(cross_output))
            feed_forward = layer.linear2(layer.dropout(layer.activation(layer.linear1(x))))
            x = layer.norm3(x + layer.dropout3(feed_forward))

        if self.model.transformer.decoder.norm is not None:
            x = self.model.transformer.decoder.norm(x)
        self.position += 1
        return self.model.head(x[:, 0])

@torch.inference_mode()
def generate_cached(model, src, max_new_tokens=512, max_output_bytes=None):
    model.eval()
    cache = CachedDecoder(model, src, max_new_tokens)
    tgt = torch.full((len(src), 1), BOS, device=src.device, dtype=torch.long)
    done = torch.zeros(len(src), device=src.device, dtype=torch.bool)
    byte_stopped = done.clone()
    output_bytes = torch.zeros(len(src), device=src.device, dtype=torch.long)
    for _ in range(max_new_tokens):
        logits = cache.step(tgt[:, -1])
        logits[:, [PAD, BOS]] = -torch.inf
        token = logits.argmax(-1).masked_fill(done, PAD)
        if max_output_bytes is not None:
            overflow = (~done) & (token >= TOKEN_OFFSET) & (output_bytes + TOKEN_BYTE_LENGTHS[token] > max_output_bytes)
            # Stop BEFORE the whole overflowing BPE token; do not force a different token/EOS.
            byte_stopped |= overflow
            token = token.masked_fill(overflow, PAD)
            done |= overflow
        output_bytes += TOKEN_BYTE_LENGTHS[token]
        done |= token.eq(EOS)
        tgt = torch.cat([tgt, token[:, None]], dim=1)
        if bool(done.all()): break
    return tgt[:, 1:], byte_stopped

@torch.inference_mode()
def check_cached(rows, prefix_limit=64, greedy_cap=16):
    model.eval()
    src = source_batch([r['question'] for r in rows])
    tgt = pad_sequence([torch.tensor([BOS] + encode(r['code']) + [EOS]) for r in rows],
                       batch_first=True, padding_value=PAD).to(device)[:, :prefix_limit]
    memory = model.transformer.encoder(model.embed(src), src_key_padding_mask=src.eq(PAD))
    cache = CachedDecoder(model, src, tgt.shape[1])
    positions = {0, 1, 2, 7, 15, 31, tgt.shape[1] - 1}
    checks = []
    for t in range(tgt.shape[1]):
        actual = cache.step(tgt[:, t])
        if t in positions:
            expected = model.head(model.decode_prefix(tgt[:, :t+1], memory, src.eq(PAD))[:, -1])
            checks.append({'position': t, 'max_abs_error': (actual - expected).abs().max().item(),
                'all_close': bool(torch.isclose(actual, expected, atol=LOGIT_ATOL, rtol=LOGIT_RTOL).all())})
    ordinary = model.generate(src, max_new_tokens=greedy_cap)
    cached, _ = generate_cached(model, src, max_new_tokens=greedy_cap)
    return {'positions': checks, 'atol': LOGIT_ATOL, 'rtol': LOGIT_RTOL,
        'all_close': all(c['all_close'] for c in checks),
        'greedy_equal': ordinary.shape == cached.shape and bool(torch.equal(ordinary, cached)),
        'greedy_cap': greedy_cap}
result['initial_cache_check'] = check_cached(validation[:3])
save_results()
print('CACHE_CHECK', json.dumps(result['initial_cache_check']))
assert result['initial_cache_check']['all_close'] and result['initial_cache_check']['greedy_equal'], 'KV-кеш не совпал; сохрани ZIP'


CACHE_CHECK {"positions": [{"position": 0, "max_abs_error": 8.344650268554688e-07, "all_close": true}, {"position": 1, "max_abs_error": 5.960464477539062e-07, "all_close": true}, {"position": 2, "max_abs_error": 5.960464477539062e-07, "all_close": true}, {"position": 7, "max_abs_error": 8.344650268554688e-07, "all_close": true}, {"position": 15, "max_abs_error": 1.3709068298339844e-06, "all_close": true}, {"position": 31, "max_abs_error": 4.5299530029296875e-06, "all_close": true}, {"position": 63, "max_abs_error": 5.185604095458984e-06, "all_close": true}], "atol": 0.0001, "rtol": 0.0001, "all_close": true, "greedy_equal": true, "greedy_cap": 16}


## 5. Диагностика: ответы, разнообразие и использование вопроса

Прежняя ограниченная арифметика AST/имена/повторы после generation timer. Дополнительно число разных текстов кода и частота самого распространённого — диагностика сворачивания в общий шаблон, не метрика правильности.

На каждом контроле validation64: CE с правильными вопросами и теми же gold-кодами, затем с вопросами, циклически сдвинутыми на1. Меняется только источник, веса фиксированы/eval. Разница `rotated_minus_correct` показывает чувствительность teacher forcing к вопросу; сама по себе не доказывает верную greedy-математику. Оценка вне train/inference timer, не изменяет batch/RNG/optimizer.

Также пишется норма градиента **до прежнего clipping1** на log steps; clipping/обновления не меняются. Checkpoints каждые1000(latest), отдельные2k/5k/10k/20k/30k/50k; model/AdamW/RNG/tokenizer/data hashes.


In [7]:
class Unsupported(Exception):
    pass

class Undefined(Exception):
    pass

def arithmetic(code):
    try:
        tree = ast.parse(code)
        if len(tree.body) != 1 or not isinstance(tree.body[0], ast.FunctionDef):
            raise Unsupported()
        f = tree.body[0]
        if f.name != 'solve' or f.decorator_list or f.args.args or f.args.posonlyargs or f.args.kwonlyargs or f.args.vararg or f.args.kwarg:
            raise Unsupported()
        if not all((isinstance(s, (ast.Assign, ast.Return)) for s in f.body)):
            raise Unsupported()
        if any((isinstance(n, (ast.Call, ast.Attribute, ast.Subscript, ast.NamedExpr)) for n in ast.walk(f))):
            raise Unsupported()
        env = {}
        visits = 0

        def number(v):
            if type(v) not in (int, float) or not math.isfinite(v) or abs(v) > 1000000000000000.0:
                raise Unsupported()
            return v

        def expr(n):
            nonlocal visits
            visits += 1
            if visits > 2000:
                raise Unsupported()
            if isinstance(n, ast.Constant):
                return number(n.value)
            if isinstance(n, ast.Name):
                if n.id not in env:
                    raise Undefined(n.id)
                return env[n.id]
            if isinstance(n, ast.UnaryOp):
                v = expr(n.operand)
                if isinstance(n.op, ast.USub):
                    return number(-v)
                if isinstance(n.op, ast.UAdd):
                    return v
            if isinstance(n, ast.BinOp):
                a, b = (expr(n.left), expr(n.right))
                if isinstance(n.op, ast.Add):
                    return number(a + b)
                if isinstance(n.op, ast.Sub):
                    return number(a - b)
                if isinstance(n.op, ast.Mult):
                    return number(a * b)
                if isinstance(n.op, ast.Div):
                    return number(a / b)
                if isinstance(n.op, ast.FloorDiv):
                    return number(a // b)
                if isinstance(n.op, ast.Mod):
                    return number(a % b)
                if isinstance(n.op, ast.Pow) and abs(b) <= 12:
                    return number(a ** b)
            raise Unsupported()
        for s in f.body:
            if isinstance(s, ast.Assign):
                if not all((isinstance(n, ast.Name) for n in s.targets)):
                    raise Unsupported()
                v = expr(s.value)
                for n in s.targets:
                    env[n.id] = v
            else:
                return {'status': 'numeric', 'value': expr(s.value)}
        return {'status': 'no_return'}
    except Undefined as e:
        return {'status': 'undefined_name', 'name': str(e)}
    except (SyntaxError, ValueError, RecursionError):
        return {'status': 'syntax_error'}
    except (ZeroDivisionError, OverflowError):
        return {'status': 'arithmetic_error'}
    except Unsupported:
        return {'status': 'unsupported'}

def repetition_properties(code, ids):
    # Heuristics only: repeated token-tail and identical consecutive nonempty lines.
    ids = [i for i in ids if i >= TOKEN_OFFSET]
    tail = 0
    for period in range(1, min(16, len(ids) // 4) + 1):
        motif = ids[-period:]
        copies = 1
        while len(ids) >= (copies + 1) * period and ids[-(copies + 1) * period:-copies * period] == motif:
            copies += 1
        if copies >= 4: tail = max(tail, copies * period)
    longest, run, previous = 0, 0, None
    for line in (line.strip() for line in code.splitlines() if line.strip()):
        run = run + 1 if line == previous else 1
        previous, longest = line, max(longest, run)
    try:
        identifiers = [a.id for a in ast.walk(ast.parse(code)) if isinstance(a, ast.Name)]
        identifier_chars = max(map(len, identifiers), default=0)
    except (SyntaxError, ValueError, RecursionError): identifier_chars = None
    return {'repeated_tail_tokens': tail, 'repeated_tail_32_plus': tail >= 32,
        'max_consecutive_identical_lines': longest, 'consecutive_lines_3_plus': longest >= 3,
        'max_identifier_characters': identifier_chars}

def extra_properties(example):
    numeric = arithmetic(example['generated_code'])
    extra = {'numeric_ast': numeric, **repetition_properties(example['generated_code'], example['generated_token_ids'])}
    if 'reference_code' in example:
        reference = arithmetic(example['reference_code'])
        comparable = numeric['status'] == reference['status'] == 'numeric'
        extra.update(reference_numeric_ast=reference, numeric_ast_comparable=comparable,
            numeric_ast_match=comparable and math.isclose(numeric['value'], reference['value'], rel_tol=1e-7, abs_tol=1e-7))
    return extra

def compare_saved09(step, name, summary):
    prior = result['historical_wide09']['diagnostics'].get(str(step), {}).get(name)
    if prior is None: return
    keys = ['bpe_loss', 'seconds', 'parses_as_python_count', 'solve_returns_value_count',
            'eos_emitted_count', 'token_cap_without_eos_count', 'numeric_ast_comparable_count',
            'numeric_ast_match_count', 'numeric_ast_undefined_name_count']
    diversity = result['historical_wide09']['diversity'].get(f'{step}/{name}', {})
    print('COMPARE_SAVED09', json.dumps({'step': step, 'split': name,
        'saved09': {k: prior[k] for k in keys}, 'current10': {k: summary[k] for k in keys},
        'saved09_diversity': {k: diversity.get(k) for k in ['unique_generated_code_count', 'most_common_generated_code_count', 'most_common_fraction']},
        'current10_diversity': {k: summary[k] for k in ['unique_generated_code_count', 'most_common_generated_code_count', 'most_common_generated_code_fraction']},
        'scope': 'historical T4 numbers plus static diagnostics; same model/tokenizer/pairs, LR differs; numeric eligibility varies, not pass@1'}), flush=True)

@torch.inference_mode()
def evaluate_loss(rows, source_questions=None):
    model.eval()
    if source_questions is not None: assert len(source_questions) == len(rows)
    loss_sum, label_count = 0.0, 0
    for offset in range(0, len(rows), 16):
        src, tgt, labels = training_batch(rows[offset:offset + 16])
        if source_questions is not None: src = source_batch(source_questions[offset:offset + 16])
        logits = model(src, tgt)
        loss_sum += F.cross_entropy(logits.reshape(-1, VOCAB), labels.reshape(-1),
            ignore_index=PAD, reduction='sum').item()
        label_count += labels.ne(PAD).sum().item()
    return loss_sum / label_count

@torch.inference_mode()
def check_question_conditioning(rows, correct_loss=None):
    assert len(rows) > 1
    questions = [row['question'] for row in rows]
    rotated = questions[1:] + questions[:1]
    correct = evaluate_loss(rows) if correct_loss is None else correct_loss
    shuffled = evaluate_loss(rows, source_questions=rotated)
    return {'examples': len(rows), 'correct_bpe_loss': correct, 'rotated_questions_bpe_loss': shuffled,
            'rotated_minus_correct': shuffled - correct,
            'scope': 'fixed rotation by1; only source questions change, gold codes/masks/weights unchanged; eval mode; sensitivity, not pass@1'}


def code_properties(code):
    def scoped_nodes(node):
        stack = [node]
        while stack:
            current = stack.pop()
            yield current
            if not isinstance(current, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef, ast.Lambda)):
                stack.extend(ast.iter_child_nodes(current))
    try:
        tree = ast.parse(code)
        functions = [n for n in tree.body if isinstance(n, ast.FunctionDef) and n.name == 'solve']
        has_return = any(isinstance(n, ast.Return) and n.value is not None
            for f in functions for stmt in f.body for n in scoped_nodes(stmt))
        return {'parses_as_python': True, 'defines_solve': bool(functions), 'solve_returns_value': has_return,
                'syntax_error_type': None, 'syntax_error_message': None, 'indentation_error': False}
    except (SyntaxError, ValueError, RecursionError) as error:
        return {'parses_as_python': False, 'defines_solve': False, 'solve_returns_value': False,
                'syntax_error_type': type(error).__name__, 'syntax_error_message': str(error),
                'indentation_error': isinstance(error, IndentationError)}
def inspect_ids(row, ids, byte_stopped=False):
    active = [i for i in ids if i != PAD]
    if EOS in active: active = active[:active.index(EOS) + 1]
    raw = decode_bytes(active)
    try:
        code = raw.decode('utf-8')
        valid_utf8 = True
    except UnicodeDecodeError:
        code, valid_utf8 = raw.decode('utf-8', errors='replace'), False
    eos = EOS in active
    example = {'question': row['question'], 'generated_code': code, 'generated_token_ids': active,
        'generated_tokens_including_eos': len(active), 'generated_utf8_bytes': len(raw), 'valid_utf8': valid_utf8,
        'eos_emitted': eos, 'reached_token_cap': len(active) == GENERATION_CAP,
        'token_cap_without_eos': len(active) == GENERATION_CAP and not eos,
        'byte_budget_stop': bool(byte_stopped),
        'stop_reason': 'eos' if eos else ('byte_budget' if byte_stopped else 'token_cap'),
        **code_properties(code)}
    if 'code' in row:
        example.update({'reference_code': row['code'], 'reference_utf8_bytes': len(row['code'].encode()),
            'reference_bpe_tokens_including_eos': len(encode(row['code'])) + 1,
            'exact_code_match': valid_utf8 and code == row['code'],
            'exact_tokens_with_eos': active == encode(row['code']) + [EOS]})
    return example

@torch.inference_mode()
def inspect_split(name, rows, max_output_bytes=None, with_loss=True):
    model.eval()
    fixed_loss = evaluate_loss(rows) if with_loss else None
    # Warm-up is outside generation timing and never changes weights.
    generate_cached(model, source_batch([rows[0]['question']]), max_new_tokens=8)
    sync()
    if device.type == 'cuda': torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    examples = []
    for offset in range(0, len(rows), INFERENCE_BATCH):
        batch = rows[offset:offset + INFERENCE_BATCH]
        tokens, stopped = generate_cached(model, source_batch([r['question'] for r in batch]),
            max_new_tokens=GENERATION_CAP, max_output_bytes=max_output_bytes)
        examples.extend(inspect_ids(row, ids, stop) for row, ids, stop
                        in zip(batch, tokens.cpu().tolist(), stopped.cpu().tolist()))
    sync()
    seconds = time.perf_counter() - started
    extra_started = time.perf_counter()
    for example in examples: example.update(extra_properties(example))
    codes = Counter(e['generated_code'] for e in examples)
    extra_seconds = time.perf_counter() - extra_started
    summary = {'split': name, 'examples': len(rows), 'bpe_loss': fixed_loss, 'seconds': seconds,
        'answers_per_second': len(rows) / seconds,
        'generated_tokens_including_eos': sum(e['generated_tokens_including_eos'] for e in examples),
        'generated_utf8_bytes': sum(e['generated_utf8_bytes'] for e in examples),
        'mean_tokens_including_eos': sum(e['generated_tokens_including_eos'] for e in examples) / len(rows),
        'mean_utf8_bytes': sum(e['generated_utf8_bytes'] for e in examples) / len(rows),
        'max_new_tokens': GENERATION_CAP, 'max_output_bytes': max_output_bytes,
        'batch_size_inference': INFERENCE_BATCH, 'peak_gpu_memory_mib':
            torch.cuda.max_memory_allocated() / 2**20 if device.type == 'cuda' else None,
        'timer_scope': 'same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics',
        'extra_diagnostics_seconds_excluded': extra_seconds,
        'unique_generated_code_count': len(codes),
        'most_common_generated_code_count': codes.most_common(1)[0][1],
        'most_common_generated_code_fraction': codes.most_common(1)[0][1] / len(examples),
        'numeric_ast_scope': 'restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1',
        'numeric_ast_statuses': dict(Counter(e['numeric_ast']['status'] for e in examples)),
        'numeric_ast_undefined_name_count': sum(e['numeric_ast']['status'] == 'undefined_name' for e in examples),
        'repeated_tail_32_plus_count': sum(e['repeated_tail_32_plus'] for e in examples),
        'consecutive_lines_3_plus_count': sum(e['consecutive_lines_3_plus'] for e in examples),
        'syntax_error_types': dict(Counter(e['syntax_error_type'] for e in examples if e['syntax_error_type']))}
    for metric in ['parses_as_python', 'defines_solve', 'solve_returns_value', 'indentation_error', 'valid_utf8',
                   'eos_emitted', 'reached_token_cap', 'token_cap_without_eos', 'byte_budget_stop']:
        summary[metric + '_count'] = sum(e[metric] for e in examples)
    if with_loss:
        summary['numeric_ast_comparable_count'] = sum(e['numeric_ast_comparable'] for e in examples)
        summary['numeric_ast_match_count'] = sum(e['numeric_ast_match'] for e in examples)
        summary['reference_numeric_ast_statuses'] = dict(Counter(e['reference_numeric_ast']['status'] for e in examples))
        for metric in ['exact_code_match', 'exact_tokens_with_eos']:
            summary[metric + '_count'] = sum(e[metric] for e in examples)
    summary['bpe_tokens_per_second'] = summary['generated_tokens_including_eos'] / seconds
    summary['output_bytes_per_second'] = summary['generated_utf8_bytes'] / seconds
    print('DIAGNOSTICS', json.dumps(summary), flush=True)
    for e in examples[:3]: print('EXAMPLE', name, json.dumps(e, ensure_ascii=False), flush=True)
    return {'summary': summary, 'examples': examples}

def save_checkpoint(step):
    state = {'model': {k: v.detach().cpu() for k, v in model.state_dict().items()},
        'optimizer': optimizer.state_dict(), 'scheduler': scheduler.state_dict(),
        'steps': step, 'config': config,
        'data_revision': revision, 'train_sha256': train_hash, 'validation_sha256': validation_hash,
        'tokenizer_json': TOKENIZER_FILE.read_text(encoding='utf-8'), 'tokenizer_sha256': tokenizer_hash,
        'python_random_state': random.getstate(), 'torch_rng_state': torch.get_rng_state(),
        'cuda_rng_state_all': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}
    temporary = LATEST_CHECKPOINT.with_suffix('.pt.tmp')
    torch.save(state, temporary)
    temporary.replace(LATEST_CHECKPOINT)
    result['latest_checkpoint_step'] = step
    if step in EVALUATION_STEPS:
        name = f'step-{step:05d}.pt'
        target = RUN_DIR / name
        temporary = target.with_suffix('.pt.tmp')
        torch.save(state, temporary)
        temporary.replace(target)
        result['checkpoint_files'][str(step)] = name
    save_results()


## 6. Обучение с нуля до50k: cosine0.001→0.0001

Прежние train64/validation64 и контрольные шаги, сравнение с09. Scheduler уменьшается плавно от прежнего начальногоLR; warmup отсутствует. Бюджет50k остаётся фиксированным даже при ранней оценке на10k. Изменяется только LR policy; дополнительных обучающих данных и архитектурных блоков нет.

LR, норма градиента до clipping1, loss и чувствительность к вопросу сохраняются. Train timer включает scheduler; оценки и checkpointing исключены. Не выбиратьcheckpoint только по AST/EOS: в09 одинаковые нерабочие программы выглядели синтаксически хорошо. Дополнительная диагностика не включает автоматическую раннюю остановку. После прерывания сохранитьZIP; повторный train поверх optimizer заблокирован. Для будущего продолжения checkpoints содержат model/AdamW/scheduler/RNG, но автоматическое продолжение этим notebook не выполняется.


In [8]:
assert not optimizer.state and result['actual_steps'] == 0, 'Train уже запускался; сохрани файлы, не запускай эту ячейку повторно'
assert scheduler.last_epoch == 0, 'Scheduler уже использовался'
assert next(model.parameters()).dtype == torch.float32
result['phase'] = 'training'
result['training_timer_scope'] = 'batch encoding/construction, transfers, forward/backward, clipping, AdamW, scheduler, block-end CUDA sync'
result['training_log'].append({'step': 0, 'train_batch_loss': None,
    'validation_loss_16': evaluate_loss(validation[:16]),
    'lr_next': optimizer.param_groups[0]['lr'], 'training_seconds': 0.0})
save_results()
run_started = time.perf_counter()
training_seconds = 0.0
completed_steps = 0
sync()
block_started, block_active = time.perf_counter(), True
try:
    for step in range(1, TOTAL_STEPS + 1):
        model.train()
        src, tgt, labels = training_batch(random.sample(train, BATCH_SIZE))
        logits = model(src, tgt)
        loss = criterion(logits.reshape(-1, VOCAB), labels.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        gradient_norm_before_clip = nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        lr_used = optimizer.param_groups[0]['lr']
        optimizer.step()
        scheduler.step()
        completed_steps = step
        if step == 1 or step % LOG_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
            sync()
            training_seconds += time.perf_counter() - block_started
            block_active = False
            result['actual_steps'] = completed_steps
            result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
            train_loss, val_loss = loss.item(), evaluate_loss(validation[:16])
            if not math.isfinite(train_loss) or not math.isfinite(val_loss):
                raise FloatingPointError('Non-finite loss; last finite checkpoint is kept')
            entry = {'step': step, 'train_batch_loss': train_loss, 'validation_loss_16': val_loss,
                'gradient_norm_before_clip': float(gradient_norm_before_clip),
                'lr_used': lr_used, 'lr_next': optimizer.param_groups[0]['lr'],
                'training_seconds': training_seconds, 'updates_per_second': step / training_seconds,
                'mean_pair_presentations': step * BATCH_SIZE / len(train)}
            result['training_log'].append(entry)
            print('LOSS', json.dumps(entry), flush=True)
            save_results()
            if step in EVALUATION_STEPS:
                parity = check_cached(validation[:3])
                result['diagnostics'][str(step)] = {'cache_check': parity}
                save_results()
                if not parity['all_close'] or not parity['greedy_equal']:
                    raise RuntimeError('KV parity failed; download results instead of continuing')
                for name, rows in [('train64', train_diagnostic), ('validation64', validation)]:
                    result['diagnostics'][str(step)][name] = inspect_split(name, rows)
                    save_results()
                    compare_saved09(step, name, result['diagnostics'][str(step)][name]['summary'])
                sensitivity = check_question_conditioning(validation,
                    correct_loss=result['diagnostics'][str(step)]['validation64']['summary']['bpe_loss'])
                result['diagnostics'][str(step)]['validation64_conditioning'] = sensitivity
                print('QUESTION_CONDITIONING', step, json.dumps(sensitivity), flush=True)
                save_results()
            if step % CHECKPOINT_EVERY == 0 or step in EVALUATION_STEPS or step == TOTAL_STEPS:
                save_checkpoint(completed_steps)
                print('CHECKPOINT', completed_steps, flush=True)
            sync()
            if step < TOTAL_STEPS:
                block_started, block_active = time.perf_counter(), True
    result['run_finished'] = True
    result['phase'] = 'training_finished'
except BaseException as error:
    result['phase'] = 'training_stopped'
    result['training_error'] = {'type': type(error).__name__, 'message': str(error)}
    raise
finally:
    sync()
    if block_active: training_seconds += time.perf_counter() - block_started
    result['actual_steps'] = completed_steps
    result['training_seconds_excluding_evaluation_logging_and_checkpoints'] = training_seconds
    result['active_wall_seconds_including_diagnostics_and_checkpoints'] = time.perf_counter() - run_started
    if result.get('training_error', {}).get('type') != 'FloatingPointError':
        save_checkpoint(completed_steps)
    save_results()


LOSS {"step": 1, "train_batch_loss": 8.548746109008789, "validation_loss_16": 7.232041828610832, "gradient_norm_before_clip": 5.85676908493042, "lr_used": 0.001, "lr_next": 0.0009999999991117355, "training_seconds": 0.31340125400004126, "updates_per_second": 3.190797698594621, "mean_pair_presentations": 0.0004}
LOSS {"step": 50, "train_batch_loss": 4.272313117980957, "validation_loss_16": 3.8714916722824726, "gradient_norm_before_clip": 4.581298828125, "lr_used": 0.0009999978672788669, "lr_next": 0.0009999977793408335, "training_seconds": 1.6456743150000648, "updates_per_second": 30.38268237175351, "mean_pair_presentations": 0.02}
LOSS {"step": 100, "train_batch_loss": 3.1827638149261475, "validation_loss_16": 3.5768379356820463, "gradient_norm_before_clip": 2.241884469985962, "lr_used": 0.0009999912941487203, "lr_next": 0.0009999911173852569, "training_seconds": 3.0622509710001395, "updates_per_second": 32.65571664341402, "mean_pair_presentations": 0.04}
LOSS {"step": 150, "train_batc

## 7. Дополнительная validation и прежний byte-limit

При byte-limit512 прекращаем генерацию **перед** BPE-токеном, который переполнил бы бюджет; не подменяем его EOS или другим токеном. Поэтому фактическая длина может быть немного меньше512 байтов. Это диагностика покрытия, а не идеально одинаковый поиск между разными токенизациями.


In [9]:
assert result['run_finished'] and result['actual_steps'] == TOTAL_STEPS
result['phase'] = 'final_validation'
result['final_validation'] = {}
for name, rows, byte_limit in [('validation192_extra', validation_extra, None),
                              ('validation64_byte512', validation, BYTE_BUDGET)]:
    result['final_validation'][name] = inspect_split(name, rows, max_output_bytes=byte_limit)
    save_results()
last = result['diagnostics'][str(TOTAL_STEPS)]['validation64']['summary']
print('VALIDATION64 versus saved wide09:')
compare_saved09(TOTAL_STEPS, 'validation64', last)
print('Same tokenizer: BPE token-loss is comparable with saved08/09, not with byte-loss.')
result['phase'] = 'validation_finished'
save_results()


DIAGNOSTICS {"split": "validation192_extra", "examples": 192, "bpe_loss": 1.8373017074163205, "seconds": 0.7313324990000183, "answers_per_second": 262.53448364803927, "generated_tokens_including_eos": 7208, "generated_utf8_bytes": 26736, "mean_tokens_including_eos": 37.541666666666664, "mean_utf8_bytes": 139.25, "max_new_tokens": 512, "max_output_bytes": null, "batch_size_inference": 32, "peak_gpu_memory_mib": 214.0576171875, "timer_scope": "same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics", "extra_diagnostics_seconds_excluded": 0.05151374599995506, "unique_generated_code_count": 12, "most_common_generated_code_count": 77, "most_common_generated_code_fraction": 0.4010416666666667, "numeric_ast_scope": "restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1", "numeric_ast_statuses": {"undefine

## 8. Полный test: все 1919 вопросов и таймер T4

Test скачивается после обучения, BPE на нём не обучается. Порядок строк исходный. Даже если получится дольше180 секунд, замер завершается полностью и сохраняется. Это не финальная посылка преподавателю.


In [10]:
assert result['run_finished'] and result['actual_steps'] == TOTAL_STEPS
result['phase'] = 'test_data'
save_results()
test_file = cache / 'test.parquet'
if not test_file.exists(): urlretrieve(f'{base_url}/data/test/test.parquet', test_file)
assert sha(test_file.read_bytes()) == '8bde59c8d8bb7edc4e8a503be22fc132c4a47773025d333432e78c5489e6522a'
test_questions = pq.read_table(test_file, columns=['question'])['question'].to_pylist()
assert len(test_questions) == 1919
assert all(len(encode(q)) + 2 <= MAX_POSITIONS for q in test_questions)
test_rows = [{'question': q} for q in test_questions]
result['test_data'] = {'rows': len(test_rows), 'questions_sha256': data_hash(test_rows),
    'file_sha256': sha(test_file.read_bytes()), 'row_order': 'unchanged parquet order', 'dataset_revision': revision}
result['phase'] = 'test_generation'
save_results()
test_output = inspect_split('test1919', test_rows, with_loss=False)
TEST_GENERATIONS = RUN_DIR / 'test-generations.json'
write_json(TEST_GENERATIONS, test_output['examples'])
result['test'] = {'summary': test_output['summary'], 'examples_file': TEST_GENERATIONS.name}
result['test']['summary']['within_180_seconds'] = test_output['summary']['seconds'] <= 180
result['test_evaluated'] = True
result['phase'] = 'complete'
save_results()
print('TEST_SUMMARY', json.dumps(result['test']['summary']), flush=True)


DIAGNOSTICS {"split": "test1919", "examples": 1919, "bpe_loss": null, "seconds": 7.296210002999942, "answers_per_second": 263.01326294212686, "generated_tokens_including_eos": 73045, "generated_utf8_bytes": 269532, "mean_tokens_including_eos": 38.064095883272536, "mean_utf8_bytes": 140.45440333507034, "max_new_tokens": 512, "max_output_bytes": null, "batch_size_inference": 32, "peak_gpu_memory_mib": 222.17626953125, "timer_scope": "same as08: encode, transfers, encoder/KV decode, CPU decoding and AST; excludes loss, warmup, data loading and extra arithmetic/repetition diagnostics", "extra_diagnostics_seconds_excluded": 0.3499361270000918, "unique_generated_code_count": 19, "most_common_generated_code_count": 701, "most_common_generated_code_fraction": 0.36529442417926006, "numeric_ast_scope": "restricted straight-line numeric AST; no exec/eval/calls/imports/conditions/loops/strings; unsupported not scored; reference may be wrong; not pass@1", "numeric_ast_statuses": {"arithmetic_error"

## 9. Скачать архив — также после остановки предыдущей ячейки

ZIP содержит tokenizer, JSON с индивидуальными validation-генерациями, test-генерации, latest и все контрольные checkpoints. Сам notebook с outputs скачай отдельно через меню Colab.


In [11]:
# This cell never trains, generates or restarts anything.
assert RESULTS_FILE.exists(), 'Сначала должны появиться результаты запуска'
result['archive_prepared'] = True
save_results()
README_FILE = RUN_DIR / 'README.txt'
README_FILE.write_text(
    'Experiment10: cosine LR0.001 to0.0001 over50k optimizer updates, no warmup/restarts; same wide256/BPE08/original20k, new weights.\n'
    'results.json contains phase/completed steps/validation examples and test summary.\n'
    'tokenizer.json has4093 content IDs; model adds offset3 and PAD/BOS/EOS0/1/2.\n'
    'latest.pt and step-*.pt include model, optimizer, tokenizer JSON, data hashes and RNG.\n'
    'test-generations.json exists only after complete test. No exec/eval/calls/imports; restricted numeric AST checks are not official pass@1.\n'
    'For analysis return this ZIP and the IPYNB downloaded with outputs.\n', encoding='utf-8')
files_to_pack = sorted(p for p in RUN_DIR.iterdir() if p.is_file()
                       and p.name != 'manifest.json' and not p.name.endswith('.tmp'))
manifest = {'experiment': result['experiment'], 'actual_steps': result['actual_steps'], 'phase': result['phase'],
    'files': [{'name': p.name, 'bytes': p.stat().st_size, 'sha256': sha(p.read_bytes())} for p in files_to_pack]}
write_json(RUN_DIR / 'manifest.json', manifest)
with zipfile.ZipFile(ARCHIVE_FILE, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for p in files_to_pack + [RUN_DIR / 'manifest.json']: archive.write(p, arcname=p.name)
print('ARCHIVE', str(ARCHIVE_FILE), ARCHIVE_FILE.stat().st_size, 'bytes', '| steps:', result['actual_steps'])
from google.colab import files
files.download(str(ARCHIVE_FILE))


ARCHIVE hw02-bpe4k-width256-cosine-20k-results.zip 465641025 bytes | steps: 50000


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>